# SpaceX Data Collection
**Purpose:** retrieve the course-provided SpaceX launch JSON and inspect its structure before extracting analysis fields.

This notebook uses the IBM Skills Network course dataset. Run the cells in order. If the hosted dataset is temporarily unavailable, check the URL before substituting another source.

In [ ]:
import requests
import pandas as pd
from IPython.display import display

DATA_URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json"
response = requests.get(DATA_URL, timeout=30)
response.raise_for_status()
raw = response.json()
print("Records retrieved:", len(raw))
print("First record keys:", list(raw[0].keys()) if raw else "No records")

## Inspect nested fields
The source contains nested launch, payload and core fields. Inspecting a record first helps avoid assuming that all launches have the same nested values.

In [ ]:
display(pd.json_normalize(raw).head())

## Create a launch-level table
This extraction is defensive: it handles empty lists and missing fields. The course dataset can differ from the live SpaceX API schema, so review the preview and missing-value summary.

In [ ]:
def first_item(value):
    return value[0] if isinstance(value, list) and value else {}

rows = []
for item in raw:
    payload = first_item(item.get("payloads"))
    core = first_item(item.get("cores"))
    rows.append({
        "FlightNumber": item.get("flight_number"),
        "Date": item.get("date_utc"),
        "Name": item.get("name"),
        "PayloadMass": payload.get("mass_kg") if isinstance(payload, dict) else None,
        "Orbit": payload.get("orbit") if isinstance(payload, dict) else None,
        "Outcome": core.get("landing_success") if isinstance(core, dict) else None,
        "Flights": core.get("flight") if isinstance(core, dict) else None,
        "GridFins": core.get("gridfins") if isinstance(core, dict) else None,
        "Reused": core.get("reused") if isinstance(core, dict) else None,
        "LandingPad": core.get("landpad") if isinstance(core, dict) else None,
        "BoosterSerial": core.get("core") if isinstance(core, dict) else None
    })
launch_df = pd.DataFrame(rows)
display(launch_df.head())
print("Shape:", launch_df.shape)
display(launch_df.isna().sum().to_frame("missing_values"))

In [ ]:
launch_df.to_csv("spacex_api_launches.csv", index=False)
print("Saved: spacex_api_launches.csv")